# LangChain Client + API Demo
This notebook demonstrates how to call a remote LangServe API and how such an API is built on the server-side.

## 1. Install Required Client-Side Packages
These packages are required to interact with LangServe APIs.

In [2]:
# !pip install langchain -q
# !pip install langchain_mistralai -q
# !pip install langserve -q
# !pip install fastapi -q

## 2. Call the Remote LangServe API
Use RemoteRunnable to call the translation API exposed on Hugging Face Spaces.

In [ ]:
from langserve import RemoteRunnable

HOST = "https://antoinekrajnc-sample-langchain-api.hf.space"
ENDPOINT = "/chain/"

translator = RemoteRunnable(f"{HOST}{ENDPOINT}")

response = translator.invoke({
    "language": "French",
    "text": "What is the name of the most famous Star Wars bounty hunter?"
})

print(response)

# invoke = envoyer des prompts à des modèles 

Quel est le nom du **chasseur de primes** le plus célèbre de *Star Wars* ?

*(Réponse : Boba Fett)* 😊


## 3. Stream the Response Like ChatGPT
See the result token by token.

In [ ]:
for token in translator.stream({
    "language": "French",
    "text": "What is the name of the most Star Wars bounty hunter?"
}):
    print(token, end='', flush=True)

Quel est le nom du **chasseur de primes** le plus célèbre de *Star Wars* ?

*(If you're referring to the most iconic one, the answer is likely **Boba Fett**—but others like **Jango Fett**, **IG-88**, or **Bossk** are also well-known.)* Would you like a full list?

## 4. API-Side Chain Definition (Server-Side Code)
This is how the translation chain is defined on the server.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_mistralai import ChatMistralAI

system_template = "Translate the following into {language}:"
prompt_template = ChatPromptTemplate.from_messages([
    ('system', system_template),
    ('user', '{text}')
])

model = ChatMistralAI(model="mistral-large-latest")
parser = StrOutputParser()

pipe_sequence = prompt_template.pipe(model).pipe(parser)


## 5. Expose the Chain with LangServe
The chain is exposed as an API route using FastAPI and LangServe.

In [ ]:
from fastapi import FastAPI
from langserve import add_routes

app = FastAPI(
    title="Sample LangServe API",
    version="0.1",
    description="Simple FastAPI app that integrates LangServe"
)

add_routes(app, pipe_sequence, "/translate")

## 6. Prompt Templates Overview
### 6.1 PromptTemplate

In [4]:
from langchain_core.prompts import PromptTemplate

# 1. Define a template with a variable {name}
prompt_template = PromptTemplate.from_template("Hello {name}")

# 2. Fill the template with a value for 'name'
prompt_result = prompt_template.invoke({"name": "Jedha"})

# 3. Convert the result to string
print(prompt_result.to_string())


Hello Jedha


### 6.2 ChatPromptTemplate

In [5]:
from langchain_core.prompts import ChatPromptTemplate

# Define a chat prompt template with multiple roles (system + user here)
template = ChatPromptTemplate.from_messages([
    # The system message sets the role and behavior of the AI
    ("system", "You are a protocol droid. Your name is {name}."),
    # The user message will be filled with whatever the human asks
    ("user", "{user_input}")
])

# Fill in the placeholders {name} and {user_input} with actual values
prompt_value = template.invoke({
    "name": "R-3PO",                      # placeholder {name}
    "user_input": "What is your primary function?"  # placeholder {user_input}
})

# Convert the prompt to messages, ready to be sent to the LLM
print(prompt_value.to_messages())


[SystemMessage(content='You are a protocol droid. Your name is R-3PO.', additional_kwargs={}, response_metadata={}), HumanMessage(content='What is your primary function?', additional_kwargs={}, response_metadata={})]



## 📝 Prompt Templates Overview

LangChain provides **different classes** to **structure the instructions you send to an LLM**.

These **prompt templates** are **defined by you, the developer**.  
The **user only provides the input parameters** (like the text to translate, the name of a character, etc.).

The goal of this section is to show you the **types of prompt templates** you can choose from when building your application.

| **Prompt Type**            | **When to Use It**                                                      |
|---------------------------|-------------------------------------------------------------------------|
| `PromptTemplate`           | For simple, one-shot instructions like _"Translate this text to French"_.|
| `ChatPromptTemplate`       | For simulating chat interactions with roles (system, user, assistant).   |
| `MessagesPlaceholder`      | For adding chat history (memory) into the conversation.                  |

### 🟢 Example Recap

- **PromptTemplate**  
  _One sentence with placeholders._
  ```python
  "Tell me about {character}"
  ```

- **ChatPromptTemplate**  
  _Simulated chat with roles: system, user, assistant._
  ```python
  [
    ("system", "You are a Star Wars expert."),
    ("user", "Tell me about Luke Skywalker."),
    ("assistant", "Luke is a Jedi Knight...")
  ]
  ```

- **MessagesPlaceholder**  
  _Keeps memory of previous messages to build context over time._

### ✅ Why It Matters
Choosing the **right prompt structure** helps your LLM:
- Understand the task better.
- Generate more relevant and coherent responses.
- Support conversational memory when needed.


### 6.3 MessagesPlaceholder

In [ ]:
from langchain_core.prompts import MessagesPlaceholder
from langchain_core.prompts import ChatPromptTemplate

# Step 1: Create a placeholder for message history
prompt = MessagesPlaceholder("history", optional=True)  # Leave optional=True in case there are no previous messages.

# Step 2: Add some initial messages to simulate past interaction
prompt.format_messages(
    history=[
        ("system", "You are a protocol droid designed to assist sentient beings."),
        ("human", "Greetings, droid."),
    ]
)

# Step 3: Create a new ChatPromptTemplate with MessagesPlaceholder for the conversation history
prompt = ChatPromptTemplate.from_messages(
    [
        ("system", "You are a protocol droid with the designation R-3PO."),
        MessagesPlaceholder("history"),  # Placeholder for message history
        ("human", "{question}")  # The human asks a new question
    ]
)

# Step 4: Invoke the prompt with message history and the new question
response = prompt.invoke(
   {
       "history": [("human", "Calculate the coordinates for the jump to lightspeed."), 
                   ("ai", "The jump coordinates are calculated: 12.345, -45.678.")],
       "question": "Now, plot a course to the nearest star system."
   }
)

# Step 5: Simulate the conversation output
print(response.to_messages())


[SystemMessage(content='You are a protocol droid with the designation R-3PO.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Calculate the coordinates for the jump to lightspeed.', additional_kwargs={}, response_metadata={}), AIMessage(content='The jump coordinates are calculated: 12.345, -45.678.', additional_kwargs={}, response_metadata={}), HumanMessage(content='Now, plot a course to the nearest star system.', additional_kwargs={}, response_metadata={})]
